# NB01 - Data Preparation and Reproducible Ingestion

## Purpose and role

This notebook constructs the reproducible data-ingestion layer for the Freddie Mac Single-Family Loan-Level Dataset (SFLLD). It discovers, validates, parses, standardizes, and stores raw origination and monthly performance files as Parquet, providing the data foundation for all downstream notebooks. Its role is infrastructural: later empirical results depend on a clearly pinned data release, documented parsing rules, reproducible file discovery, and stable output paths.

---

## Official documentation and release pinning

The notebook uses the following official Freddie Mac documentation as the source of truth for dataset scope, release metadata, file layout, field definitions, and interpretation rules:

- Freddie Mac (2026a) = *Single-Family Loan-Level Dataset Frequently Asked Questions (FAQ)*.
- Freddie Mac (2026b) = *Single-Family Loan-Level Dataset General User Guide*.
- Freddie Mac (2026c) = *Single-Family Loan-Level Dataset Release Notes*.

The data release is pinned to **Release 46**:

| Metadata item | Value |
|---|---:|
| Release number | 46 |
| Release date | 2026-01-28 |
| Origination cutoff date | 2025-09-30 |
| Performance cutoff date | 2025-09-30 |
| Total quarters | 107 |
| Approx. origination records | 48.61 million |
| Approx. performance records | 2.81 billion |

Freddie Mac describes the Standard Dataset as a **living dataset** that may be corrected or updated over time, making this release pin a required part of the empirical design (Freddie Mac, 2026b, 2026c).

---

## Dataset scope and interpretation boundary

This notebook ingests the **Standard Dataset only**: fully amortizing fixed-rate single-family mortgages acquired by Freddie Mac, closely resembling mortgages eligible for Single-Family Credit Risk Transfer (CRT) transactions (Freddie Mac, 2026b). Loan types excluded from the Standard Dataset - ARMs, government-insured loans, pre-March 2015 Home Possible mortgages, and others - are outside this analysis; the Non-Standard Dataset, last refreshed in January 2022, is not used here.

---

## Notebook structure

| Section(s) | Description |
|---|---|
| 0 | Setup & Configuration - paths, imports, runtime settings, release constants |
| 1 | Column Schemas - 32-column origination and performance column definitions with type rationale |
| 2 | File Discovery & SQL-Casting Helpers - regex patterns, output-path builder, and `TRY_CAST`/`TRY_STRPTIME` wrappers |
| 3-4 | SQL Builders - DuckDB `SELECT` clauses for origination and performance ingestion |
| 5-8 | Core Utilities - DuckDB connection, ZIP extraction, TXT→Parquet function, quarter inspection |
| 9-11 | Manifest & Ingestion - file discovery, manifest construction, pipeline execution |
| 12-14 | Validation - file counts, row counts against Release 46 reference figures, schema spot-check |
| 15 | Repartition - performance data repartitioned by reporting year for efficient downstream queries |
| 16 | Ingestion Manifest - final row-count cross-check and `ingestion_manifest.json`, read directly by NB02 and NB03 |

---
## Section 0 · Setup & Configuration

### 0.1 · Install Dependencies

In [ ]:
%pip install -q duckdb pyarrow pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 214.3 MB/s eta 0:00:00


### 0.2 · Imports

In [ ]:
from __future__ import annotations

import io
import json
import os
import re
import shutil
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import pandas as pd
from tqdm.auto import tqdm

### 0.3 · Google Drive Mount & Path Configuration

Source ZIPs live in `My Drive / master_thesis / data /`.

Local NVMe (`/content/`) is used exclusively for:
- Temporary TXT extraction (deleted immediately after each quarter)
- DuckDB spill directory (large sort/hash operations)
- Repartitioned performance output

Ephemeral work files are written to local NVMe rather than Drive.


In [ ]:
IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/master_thesis")

DATA_ROOT = DRIVE_ROOT / "data"

# Drive paths (persistent - see §0.3)
MANIFEST_DIR = DRIVE_ROOT / "manifests"
PARQUET_ROOT    = DATA_ROOT / "parquet"
ORIGINATION_OUT = PARQUET_ROOT / "origination"
PERFORMANCE_OUT = PARQUET_ROOT / "performance"

# Local NVMe paths (ephemeral - see §0.3)
LOCAL_ROOT  = Path("/content") if IS_COLAB else DRIVE_ROOT
TEMP_ROOT   = LOCAL_ROOT / "_tmp_ingest"        # extracted TXTs - deleted per quarter
DUCKDB_TEMP = LOCAL_ROOT / "duckdb_tmp"         # DuckDB spill for large ops
DUCKDB_PATH = LOCAL_ROOT / "freddie_mac_ingest.duckdb"  # ephemeral; parquets are durable

for d in [TEMP_ROOT, DUCKDB_TEMP, ORIGINATION_OUT, PERFORMANCE_OUT, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"IS_COLAB           : {IS_COLAB}")
print(f"DATA_ROOT  (ZIPs)  : {DATA_ROOT}")
print(f"ORIGINATION_OUT    : {ORIGINATION_OUT}")
print(f"PERFORMANCE_OUT    : {PERFORMANCE_OUT}")
print(f"TEMP_ROOT  (local) : {TEMP_ROOT}")
print(f"DUCKDB_TEMP(local) : {DUCKDB_TEMP}")
assert DATA_ROOT.exists(), f"Data root not found: {DATA_ROOT}"

Mounted at /content/drive
IS_COLAB           : True
DATA_ROOT  (ZIPs)  : /content/drive/MyDrive/master_thesis/data
ORIGINATION_OUT    : /content/drive/MyDrive/master_thesis/data/parquet/origination
PERFORMANCE_OUT    : /content/drive/MyDrive/master_thesis/data/parquet/performance
TEMP_ROOT  (local) : /content/_tmp_ingest
DUCKDB_TEMP(local) : /content/duckdb_tmp


### 0.4 · Runtime Settings

DuckDB connection parameters and ingestion behaviour flags, consumed by `connect_duckdb()`.



In [ ]:
# Settings consumed by connect_duckdb()
N_THREADS           = min(40, max(1, (os.cpu_count() or 4) - 4))
DUCKDB_MEMORY_LIMIT = "160GB"

OVERWRITE_EXISTING     = False
PARQUET_COMPRESSION    = "ZSTD"
PARQUET_ROW_GROUP_SIZE = 250_000

print(f"vCPUs detected      : {os.cpu_count()}")
print(f"DuckDB threads      : {N_THREADS}")
print(f"Memory limit        : {DUCKDB_MEMORY_LIMIT}")
print(f"Overwrite existing  : {OVERWRITE_EXISTING}")

vCPUs detected      : 44
DuckDB threads      : 40
Memory limit        : 160GB
Overwrite existing  : False


### 0.5 · SFLLD Release Metadata

This notebook is pinned to a specific SFLLD release. The release number and
cutoff dates identify the source snapshot; the manifest records them together
with the exact ingested row counts.

#### Why release metadata are stored in the manifest

1. **Dataset identity** - the release number identifies the SFLLD snapshot used here.
2. **Reproducibility across reruns** - the manifest records the source snapshot and exact ingested row counts.
3. **Downstream traceability** - NB02 and NB03 read and report the ingestion snapshot.

#### Population scope

Freddie Mac describes the Standard Dataset as loan-level origination, monthly performance, and actual-loss data for fully amortizing fixed-rate single-family mortgages acquired by Freddie Mac, with origination dates from 1999 through the release-specific Origination Cutoff Date. This population closely resembles mortgages eligible for Single-Family CRT transactions. The appropriate interpretation is therefore a **CRT-aligned Freddie Mac conventional fixed-rate mortgage population**.

The Standard Dataset includes:

- fully amortizing fixed-rate single-family mortgages;
- mortgages categorized as having verified or waived documentation;
- Relief Refinance mortgages;
- Home Possible mortgages funded on or after March 1, 2015.

The Non-Standard Dataset contains loan types outside the Standard Dataset scope, including:

- adjustable-rate mortgages, initial-interest mortgages, balloons, and mortgages with step rates;
- government-insured mortgages (FHA, VA, GRH, HUD Section 184);
- Home Possible mortgages funded on or before February 28, 2015;
- other affordable mortgages outside the Home Possible program;
- mortgages delivered under alternate agreements;
- mortgages for which documentation is not verified or waived;
- mortgages associated with Mortgage Revenue Bonds;
- mortgages delivered with certain non-standard credit enhancement structures.

This defines the outer source-population boundary; NB03 defines the narrower loan-month analysis population. Results should not be generalized mechanically to adjustable-rate mortgages, government-insured loans, portfolio loans, or the broader U.S. mortgage market.

Source: Freddie Mac (2026b)

In [ ]:
SFLLD_RELEASE_METADATA: dict = {
    "release_number"                         : 46,
    "release_date"                           : "2026-01-28",
    "origination_cutoff_date"                : "2025-09-30",
    "performance_cutoff_date"                : "2025-09-30",
    "total_quarters"                         : 107,
    "approx_origination_records_millions"    : 48.61,
    "approx_performance_records_billions"    : 2.81,
    "dataset"                                : "Standard",
    "user_guide_version"                     : "January 2026",
}

print("SFLLD release pinned for reproducibility:")
for k, v in SFLLD_RELEASE_METADATA.items():
    vstr = str(v)
    print(f"  {k:<45}: {vstr[:90]}{'...' if len(vstr) > 90 else ''}")

SFLLD release pinned for reproducibility:
  release_number                               : 46
  release_date                                 : 2026-01-28
  origination_cutoff_date                      : 2025-09-30
  performance_cutoff_date                      : 2025-09-30
  total_quarters                               : 107
  approx_origination_records_millions          : 48.61
  approx_performance_records_billions          : 2.81
  dataset                                      : Standard
  user_guide_version                           : January 2026


---
## Section 1 · Column Schemas

This section defines the column names and parsing types for the SFLLD origination and monthly performance files, based on the Freddie Mac *General User Guide* (January 2026). Both files contain 32 documented columns.

The parsing rule used here is **preservation-first**: fields that function as disclosure identifiers, categorical codes, or mixed alphanumeric fields are stored as `VARCHAR`, even when some observed values look numeric.

### Field-specific schema decisions

| Field | Freddie Mac documentation | Notebook parsing decision |
|---|---|---|
| `postal_code` | Origination col 19. Disclosed as `###00` (last two digits masked). Documented type: Numeric 5. | `VARCHAR` - string parsing preserves the masking format and leading zeros (e.g. `01200`). |
| `loan_sequence_number` | Origination col 20 / performance col 1. Alphanumeric identifier. | `VARCHAR` - identifier, not a numeric measurement. |
| `current_loan_delinquency_status` | Performance col 4. Alphanumeric. Numeric values are MBA delinquency buckets; `XX` = Unknown; `RA` = REO Acquisition (current); `R` = REO Acquisition (legacy, pre-Release 29). | `VARCHAR` - numeric coercion would silently drop non-numeric status codes. |
| `zero_balance_code` | Performance col 9. Termination-event codes: `01`, `02`, `03`, `09`, `15`, `16`, `96`. | `VARCHAR` - leading zeros on `01`/`02`/`03`/`09` must be preserved; these are codes, not magnitudes. |
| `net_sale_proceeds` | Performance col 15. Alphanumeric. Dollar amounts or `U = Unknown`. | `VARCHAR` - preserves the `U` special value.
| `estimated_loan_to_value` | Performance col 26. Numeric Literal Decimal. `1-998` valid; `999` = Unknown (AVM unavailable); `NULL` = data not available; April 2017+ periods only. | `DOUBLE` - genuinely numeric; treat `999.0` as a sentinel, not a valid LTV. |


Source: Freddie Mac (2026b), with historical status-code changes cross-checked against Freddie Mac (2026c).

In [ ]:
ORIGINATION_COLS = [
    "credit_score",                              # col 01  documented credit-score (FICO) range (User Guide and FAQ differ slightly at the lower bound (300 vs. <301)(Freddie Mac, 2026a)); 9999=Not Available
    "first_payment_date",                        # col 02  YYYYMM
    "first_time_homebuyer_flag",                 # col 03  Y/N/9; blanks may occur for historical/not-applicable cases
    "maturity_date",                             # col 04  YYYYMM
    "msa_or_metropolitan_division",              # col 05  5-digit MSA/Division code; NULL=neither MSA nor Division, or unknown
    "mortgage_insurance_percentage",             # col 06  1-55%; 0=no MI; 999=Not Available
    "number_of_units",                           # col 07  1-4; 99=Not Available
    "occupancy_status",                          # col 08  P/I/S/9
    "original_combined_loan_to_value",           # col 09  regime-dependent; 999=Not Available
    "original_debt_to_income_ratio",             # col 10  0<DTI<=65%; 999=DTI>65%, unavailable, or Relief Refinance masking
    "original_upb",                              # col 11  rounded to nearest $1,000
    "original_loan_to_value",                    # col 12  regime-dependent; 999=Not Available
    "original_interest_rate",                    # col 13  literal decimal
    "channel",                                   # col 14  R/B/C/T/9; B and C reliable only from 2008
    "prepayment_penalty_mortgage_flag",          # col 15  Y/N
    "amortization_type",                         # col 16  always 'FRM' in Standard Dataset
    "property_state",                            # col 17  two-letter state/territory code
    "property_type",                             # col 18  CO/PU/MH/SF/CP/99
    "postal_code",                               # col 19  VARCHAR - preserves leading zeros (e.g. '01200')
    "loan_sequence_number",                      # col 20  PYYQnXXXXXXX
    "loan_purpose",                              # col 21  P/C/N/R/9
    "original_loan_term",                        # col 22  integer months
    "number_of_borrowers",                       # col 23  semantics differ pre/post 2018Q2 - see NB02
    "seller_name",                               # col 24
    "servicer_name",                             # col 25
    "super_conforming_flag",                     # col 26  Y/blank; applies to loans originated >= 2008-10-01 and delivered >= 2009-01-01
    "pre_relief_refinance_loan_sequence_number", # col 27  populated only for Relief Refinance loans
    "special_eligibility_program",               # col 28  H/F/R/9
    "relief_refinance_indicator",                # col 29  Y/blank
    "property_valuation_method",                 # col 30  1-4/7; populated for originations >= 2017-01-01; codes redefined in Release 46
    "interest_only_indicator",                   # col 31  always 'N' in Standard Dataset
    "mi_cancellation_indicator",                 # col 32  Y/N/7/9; added in Release 35 (April 2023)
]

PERFORMANCE_COLS = [
    "loan_sequence_number",                      # col 01
    "monthly_reporting_period",                  # col 02  YYYYMM
    "current_actual_upb",                        # col 03  DOUBLE; early UPB rounding at loan_age<=6; not rounded after modification
    "current_loan_delinquency_status",           # col 04  VARCHAR - 0/1/2.../XX/'RA'/legacy 'R'; REO codes are not DPD buckets
    "loan_age",                                  # col 05  re-based to modification first-payment date; payment deferrals do not reset it
    "remaining_months_to_legal_maturity",        # col 06  uses modified maturity date for modified loans
    "defect_settlement_date",                    # col 07  YYYYMM
    "modification_flag",                         # col 08  Y=current period / P=prior period / NULL=not modified
    "zero_balance_code",                         # col 09  VARCHAR - 01/02/03/09/15/16/96 (leading zeros preserved)
    "zero_balance_effective_date",               # col 10  YYYYMM
    "current_interest_rate",                     # col 11  DOUBLE literal decimal
    "current_non_interest_bearing_upb",          # col 12  non-zero after payment deferral or principal forbearance
    "ddlpi",                                     # col 13  Due Date of Last Paid Installment; YYYYMM
    "mi_recoveries",                             # col 14  DOUBLE; populated at property disposition
    "net_sale_proceeds",                         # col 15  VARCHAR - can contain 'U' (unknown) in addition to dollar amounts
    "non_mi_recoveries",                         # col 16  DOUBLE
    "expenses",                                  # col 17  DOUBLE; sum of cols 18-21
    "legal_costs",                               # col 18  DOUBLE
    "maintenance_and_preservation_costs",        # col 19  DOUBLE
    "taxes_and_insurance",                       # col 20  DOUBLE
    "miscellaneous_expenses",                    # col 21  DOUBLE
    "actual_loss_calculation",                   # col 22  DOUBLE; NULL for defect-settlement or final-3-month dispositions
    "cumulative_modification_cost",              # col 23  DOUBLE
    "interest_rate_step_indicator",              # col 24  Y/N/NULL; renamed from Step Modification Flag in Release 45
    "payment_deferral_flag",                     # col 25  Y=current / P=prior / NULL; distinct from modification_flag since Release 25
    "estimated_loan_to_value",                   # col 26  DOUBLE ('Numeric Literal Decimal'); 999.0=Unknown; NULL=not available; only Apr-2017+ periods
    "zero_balance_removal_upb",                  # col 27  DOUBLE
    "delinquent_accrued_interest",               # col 28  DOUBLE
    "delinquency_due_to_disaster",               # col 29  Y/NULL; populated only for Jan-2014+ periods
    "borrower_assistance_status_code",           # col 30  F=Forbearance/R=Repayment/T=Trial/NULL; Jan-2014+ periods only
    "current_month_modification_cost",           # col 31  DOUBLE
    "interest_bearing_upb",                      # col 32  DOUBLE
]

assert len(ORIGINATION_COLS) == 32, f"Expected 32 origination cols, got {len(ORIGINATION_COLS)}"
assert len(PERFORMANCE_COLS) == 32, f"Expected 32 performance cols, got {len(PERFORMANCE_COLS)}"
print(f"Origination columns : {len(ORIGINATION_COLS)}")
print(f"Performance columns : {len(PERFORMANCE_COLS)}")

Origination columns : 32
Performance columns : 32


---
## Section 2 · File Patterns & SQL Helpers

Regex patterns to locate yearly ZIPs and quarterly sub-ZIPs.
SQL-expression builders for typed conversion: source fields are read as
`VARCHAR` and converted with `TRY_CAST` / `TRY_STRPTIME`, so unparseable
field values become `NULL` rather than raising conversion errors.

In [ ]:
YEAR_ZIP_PATTERN    = re.compile(r"historical_data_(\d{4})\.zip$",          re.IGNORECASE)
QUARTER_ZIP_PATTERN = re.compile(r"historical_data_(\d{4})Q([1-4])\.zip$",  re.IGNORECASE)
ORIG_TXT_PATTERN    = re.compile(r"historical_data_(\d{4})Q([1-4])\.txt$",  re.IGNORECASE)
PERF_TXT_PATTERN    = re.compile(r"historical_data_time_(\d{4})Q([1-4])\.txt$", re.IGNORECASE)


def find_year_zip_files(root: Path) -> list[Path]:
    """Return all yearly ZIP archives matching historical_data_YYYY.zip, sorted."""
    files = sorted(p for p in root.iterdir() if p.is_file() and YEAR_ZIP_PATTERN.search(p.name))
    if not files:
        raise FileNotFoundError(f"No yearly ZIP files found in {root}")
    return files


def output_path(dataset: str, year: int, quarter: str) -> Path:
    """Construct and create the output Parquet path for a given vintage."""
    base    = ORIGINATION_OUT if dataset == "origination" else PERFORMANCE_OUT
    out_dir = base / f"vintage_year={year}" / f"vintage_quarter={quarter}"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"part_{year}{quarter}.parquet"


# ── SQL type-casting helpers ────────────────────────────────────────────────
# Each function wraps its argument in safe NULL-handling before casting.

def trim_sql(expr: str) -> str:
    """NULLIF(TRIM(expr), '') - converts blank / whitespace-only fields to NULL."""
    return f"NULLIF(TRIM({expr}), '')"

def str_sql(expr: str) -> str:
    """VARCHAR: trim and null-blank. Preserves leading zeros and non-numeric codes."""
    return trim_sql(expr)

def int_sql(expr: str) -> str:
    """INTEGER: safe cast after blank-to-NULL normalisation."""
    return f"TRY_CAST({trim_sql(expr)} AS INTEGER)"

def dbl_sql(expr: str) -> str:
    """DOUBLE: safe cast preserving decimal precision."""
    return f"TRY_CAST({trim_sql(expr)} AS DOUBLE)"

def yyyymm_to_date_sql(expr: str) -> str:
    """Convert YYYYMM string to DATE (first day of that month)."""
    cleaned = trim_sql(expr)
    return f"TRY_CAST(TRY_STRPTIME({cleaned} || '01', '%Y%m%d') AS DATE)"

def build_raw_column_map(n_cols: int) -> dict[str, str]:
    """Map all n source columns to VARCHAR for raw ingestion."""
    return {f"c{i+1}": "VARCHAR" for i in range(n_cols)}

---
## Section 3 · Origination SELECT Builder

Generates the DuckDB `SELECT` clause that casts all 32 raw `VARCHAR` columns to their
target types via the helpers from §2. See §1 for
the field-by-field type rationale.

In [ ]:
def build_origination_select(
    year: int, quarter: str,
    source_year_zip: str, source_quarter_zip: str, source_txt_name: str,
) -> str:
    c = lambda i: f"c{i}"
    return f"""
    SELECT
        {int_sql(c(1))}  AS credit_score,
        {yyyymm_to_date_sql(c(2))} AS first_payment_date,
        {str_sql(c(3))}  AS first_time_homebuyer_flag,
        {yyyymm_to_date_sql(c(4))} AS maturity_date,
        {int_sql(c(5))}  AS msa_or_metropolitan_division,
        {int_sql(c(6))}  AS mortgage_insurance_percentage,
        {int_sql(c(7))}  AS number_of_units,
        {str_sql(c(8))}  AS occupancy_status,
        {int_sql(c(9))}  AS original_combined_loan_to_value,
        {int_sql(c(10))} AS original_debt_to_income_ratio,
        {dbl_sql(c(11))} AS original_upb,
        {int_sql(c(12))} AS original_loan_to_value,
        {dbl_sql(c(13))} AS original_interest_rate,
        {str_sql(c(14))} AS channel,
        {str_sql(c(15))} AS prepayment_penalty_mortgage_flag,
        {str_sql(c(16))} AS amortization_type,
        {str_sql(c(17))} AS property_state,
        {str_sql(c(18))} AS property_type,
        {str_sql(c(19))} AS postal_code,
        {str_sql(c(20))} AS loan_sequence_number,
        {str_sql(c(21))} AS loan_purpose,
        {int_sql(c(22))} AS original_loan_term,
        {int_sql(c(23))} AS number_of_borrowers,
        {str_sql(c(24))} AS seller_name,
        {str_sql(c(25))} AS servicer_name,
        {str_sql(c(26))} AS super_conforming_flag,
        {str_sql(c(27))} AS pre_relief_refinance_loan_sequence_number,
        {str_sql(c(28))} AS special_eligibility_program,
        {str_sql(c(29))} AS relief_refinance_indicator,
        {int_sql(c(30))} AS property_valuation_method,
        {str_sql(c(31))} AS interest_only_indicator,
        {str_sql(c(32))} AS mi_cancellation_indicator,

        {year}      AS vintage_year,
        '{quarter}' AS vintage_quarter,
        '{year}{quarter}' AS vintage,
        '{source_year_zip}'    AS source_year_zip,
        '{source_quarter_zip}' AS source_quarter_zip,
        '{source_txt_name}'    AS source_txt_name,
        CURRENT_TIMESTAMP      AS ingested_at_utc
    """

---
## Section 4 · Performance SELECT Builder

Generates the DuckDB `SELECT` clause for the 32 SFLLD monthly performance columns. The purpose is to preserve Freddie Mac's documented disclosure semantics during ingestion: fields with categorical codes, leading-zero event values, or mixed alphanumeric values are stored as `VARCHAR`; genuinely numeric measurement fields are stored as numeric types.

This SELECT builder preserves the raw
Freddie Mac disclosure fields. Derived variables and the 12-month delinquency
target are constructed downstream.

In [ ]:
def build_performance_select(
    year: int, quarter: str,
    source_year_zip: str, source_quarter_zip: str, source_txt_name: str,
) -> str:
    c = lambda i: f"c{i}"
    return f"""
    SELECT
        {str_sql(c(1))}  AS loan_sequence_number,
        {yyyymm_to_date_sql(c(2))} AS monthly_reporting_period,
        {dbl_sql(c(3))}  AS current_actual_upb,
        {str_sql(c(4))}  AS current_loan_delinquency_status,
        {int_sql(c(5))}  AS loan_age,
        {int_sql(c(6))}  AS remaining_months_to_legal_maturity,
        {yyyymm_to_date_sql(c(7))} AS defect_settlement_date,
        {str_sql(c(8))}  AS modification_flag,
        {str_sql(c(9))}  AS zero_balance_code,
        {yyyymm_to_date_sql(c(10))} AS zero_balance_effective_date,
        {dbl_sql(c(11))} AS current_interest_rate,
        {dbl_sql(c(12))} AS current_non_interest_bearing_upb,
        {yyyymm_to_date_sql(c(13))} AS ddlpi,
        {dbl_sql(c(14))} AS mi_recoveries,
        {str_sql(c(15))} AS net_sale_proceeds,
        {dbl_sql(c(16))} AS non_mi_recoveries,
        {dbl_sql(c(17))} AS expenses,
        {dbl_sql(c(18))} AS legal_costs,
        {dbl_sql(c(19))} AS maintenance_and_preservation_costs,
        {dbl_sql(c(20))} AS taxes_and_insurance,
        {dbl_sql(c(21))} AS miscellaneous_expenses,
        {dbl_sql(c(22))} AS actual_loss_calculation,
        {dbl_sql(c(23))} AS cumulative_modification_cost,
        {str_sql(c(24))} AS interest_rate_step_indicator,
        {str_sql(c(25))} AS payment_deferral_flag,
        {dbl_sql(c(26))} AS estimated_loan_to_value,
        {dbl_sql(c(27))} AS zero_balance_removal_upb,
        {dbl_sql(c(28))} AS delinquent_accrued_interest,
        {str_sql(c(29))} AS delinquency_due_to_disaster,
        {str_sql(c(30))} AS borrower_assistance_status_code,
        {dbl_sql(c(31))} AS current_month_modification_cost,
        {dbl_sql(c(32))} AS interest_bearing_upb,

        {year}      AS vintage_year,
        '{quarter}' AS vintage_quarter,
        '{year}{quarter}' AS vintage,
        '{source_year_zip}'    AS source_year_zip,
        '{source_quarter_zip}' AS source_quarter_zip,
        '{source_txt_name}'    AS source_txt_name,
        CURRENT_TIMESTAMP      AS ingested_at_utc
    """

---
## Section 5 · DuckDB Connection

Opens a persistent DuckDB connection.


In [ ]:
def connect_duckdb() -> duckdb.DuckDBPyConnection:
    """Open a DuckDB connection configured for the current execution environment."""
    con = duckdb.connect(str(DUCKDB_PATH))
    con.execute(f"SET threads          = {N_THREADS}")
    con.execute(f"SET memory_limit     = '{DUCKDB_MEMORY_LIMIT}'")
    con.execute(f"SET temp_directory   = '{DUCKDB_TEMP}'")
    con.execute("SET enable_progress_bar = true")
    return con

---
## Section 6 · ZIP Extraction Utility

Streams a single ZIP member to a temporary file on local NVMe. The caller is responsible for deleting the returned temp file -
see the `finally` block in `run_ingestion` (§10).

In [ ]:
def extract_member_to_temp(zf: zipfile.ZipFile, member_name: str, temp_dir: Path) -> Path:
    target = temp_dir / Path(member_name).name
    with zf.open(member_name) as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
    return target

---
## Section 7 · Core TXT → Parquet Ingestion Function

Reads one pipe-delimited TXT file via DuckDB `read_csv`, with all 32 source
columns declared as `VARCHAR`. `ignore_errors=False` keeps other parser errors
fatal, while `null_padding=True` permits missing trailing fields. The typed
SELECT builder maps field-level conversion failures to `NULL` via
`TRY_CAST` / `TRY_STRPTIME` and writes ZSTD-compressed Parquet.


In [ ]:
def ingest_txt_with_duckdb(
    con:                duckdb.DuckDBPyConnection,
    txt_path:           Path,
    out_path:           Path,
    dataset:            str,
    year:               int,
    quarter:            str,
    source_year_zip:    str,
    source_quarter_zip: str,
    source_txt_name:    str,
) -> None:
    """Read one Freddie Mac TXT file via DuckDB and write a Parquet file."""
    if out_path.exists() and not OVERWRITE_EXISTING:
        return

    raw_cols   = build_raw_column_map(32)
    select_sql = (
        build_origination_select(year, quarter, source_year_zip, source_quarter_zip, source_txt_name)
        if dataset == "origination"
        else build_performance_select(year, quarter, source_year_zip, source_quarter_zip, source_txt_name)
    )

    raw_csv = f"""
        read_csv(
            '{txt_path.as_posix()}',
            delim          = '|',
            header         = False,
            columns        = {json.dumps(raw_cols)},
            quote          = '',
            escape         = '',
            null_padding   = True,
            sample_size    = -1,
            ignore_errors  = False
        )
    """

    con.execute(f"""
        COPY (
            {select_sql}
            FROM {raw_csv}
        )
        TO '{out_path.as_posix()}'
        (
            FORMAT         PARQUET,
            COMPRESSION    {PARQUET_COMPRESSION},
            ROW_GROUP_SIZE {PARQUET_ROW_GROUP_SIZE}
        )
    """)

---
## Section 8 · Quarter ZIP Inspection

Parses a quarterly ZIP file to identify the origination and performance TXT files
and extract year/quarter metadata.


In [ ]:
def inspect_inner_quarter_zip(inner_zip_bytes: bytes, inner_zip_name: str) -> dict:
    """Inspect one quarterly ZIP and return origination/performance TXT metadata."""
    result: dict = {
        "inner_zip_name": inner_zip_name,
        "year": None, "quarter": None,
        "orig_txt": None, "perf_txt": None,
    }

    m = QUARTER_ZIP_PATTERN.search(Path(inner_zip_name).name)
    if not m:
        raise ValueError(f"Unexpected quarter ZIP name: {inner_zip_name}")

    result["year"]    = int(m.group(1))
    result["quarter"] = f"Q{m.group(2)}"

    with zipfile.ZipFile(io.BytesIO(inner_zip_bytes), "r") as qzip:
        for n in qzip.namelist():
            base = Path(n).name
            if ORIG_TXT_PATTERN.search(base):
                result["orig_txt"] = n
            elif PERF_TXT_PATTERN.search(base):
                result["perf_txt"] = n

    if result["orig_txt"] is None or result["perf_txt"] is None:
        raise ValueError(
            f"Could not identify both TXT files in {inner_zip_name}. "
            f"Found orig={result['orig_txt']}, perf={result['perf_txt']}"
        )
    return result

---
## Section 9 · Build Ingestion Manifest

Iterates over all yearly ZIPs in `DATA_ROOT`, inspects every quarterly sub-ZIP, and
records source paths plus intended output paths into a `manifest` DataFrame (one row
per quarter: source ZIP names/paths, origination/performance TXT names, and the
target Parquet paths from `output_path()`). This is a pure discovery scan and drives
the ingestion pipeline in §10.

The release-level metadata pinned in §0.5 is carried separately into
`ingestion_manifest.json`, written in §16 once ingestion and repartitioning are
complete.

In [ ]:
year_zips     = find_year_zip_files(DATA_ROOT)
manifest_rows = []

for year_zip_path in tqdm(year_zips, desc="Scanning yearly ZIPs", unit="year_zip"):
    with zipfile.ZipFile(year_zip_path, "r") as yz:
        inner_names = sorted(
            n for n in yz.namelist()
            if QUARTER_ZIP_PATTERN.search(Path(n).name)
        )
        for inner_name in inner_names:
            with yz.open(inner_name) as f:
                inner_bytes = f.read()
            meta = inspect_inner_quarter_zip(inner_bytes, inner_name)

            manifest_rows.append({
                "source_year_zip"      : year_zip_path.name,
                "source_year_zip_path" : str(year_zip_path),
                "source_quarter_zip"   : inner_name,
                "year"                 : meta["year"],
                "quarter"              : meta["quarter"],
                "orig_txt"             : meta["orig_txt"],
                "perf_txt"             : meta["perf_txt"],
                "orig_out"             : str(output_path("origination", meta["year"], meta["quarter"])),
                "perf_out"             : str(output_path("performance",  meta["year"], meta["quarter"])),
            })

manifest = (
    pd.DataFrame(manifest_rows)
    .sort_values(["year", "quarter"])
    .reset_index(drop=True)
)
manifest

Scanning yearly ZIPs:   0%|          | 0/27 [00:00<?, ?year_zip/s]

,source_year_zip,source_year_zip_path,source_quarter_zip,year,quarter,orig_txt,perf_txt,orig_out,perf_out
0,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q1.zip,1999,Q1,historical_data_1999Q1.txt,historical_data_time_1999Q1.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
1,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q2.zip,1999,Q2,historical_data_1999Q2.txt,historical_data_time_1999Q2.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
2,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q3.zip,1999,Q3,historical_data_1999Q3.txt,historical_data_time_1999Q3.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
3,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q4.zip,1999,Q4,historical_data_1999Q4.txt,historical_data_time_1999Q4.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
4,historical_data_2000.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2000Q1.zip,2000,Q1,historical_data_2000Q1.txt,historical_data_time_2000Q1.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
...,...,...,...,...,...,...,...,...,...
102,historical_data_2024.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2024Q3.zip,2024,Q3,historical_data_2024Q3.txt,historical_data_time_2024Q3.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
103,historical_data_2024.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2024Q4.zip,2024,Q4,historical_data_2024Q4.txt,historical_data_time_2024Q4.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
104,historical_data_2025.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2025Q1.zip,2025,Q1,historical_data_2025Q1.txt,historical_data_time_2025Q1.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
105,historical_data_2025.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2025Q2.zip,2025,Q2,historical_data_2025Q2.txt,historical_data_time_2025Q2.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...


### Manifest Summary

In [ ]:
display(manifest.head(12))
print(f"Total quarter ZIPs  : {len(manifest)}")
print(f"Year range          : {manifest['year'].min()} - {manifest['year'].max()}")
print("\nQuarters per year:")
print(manifest.groupby("year").size().to_string())

,source_year_zip,source_year_zip_path,source_quarter_zip,year,quarter,orig_txt,perf_txt,orig_out,perf_out
0,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q1.zip,1999,Q1,historical_data_1999Q1.txt,historical_data_time_1999Q1.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
1,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q2.zip,1999,Q2,historical_data_1999Q2.txt,historical_data_time_1999Q2.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
2,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q3.zip,1999,Q3,historical_data_1999Q3.txt,historical_data_time_1999Q3.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
3,historical_data_1999.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_1999Q4.zip,1999,Q4,historical_data_1999Q4.txt,historical_data_time_1999Q4.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
4,historical_data_2000.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2000Q1.zip,2000,Q1,historical_data_2000Q1.txt,historical_data_time_2000Q1.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
5,historical_data_2000.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2000Q2.zip,2000,Q2,historical_data_2000Q2.txt,historical_data_time_2000Q2.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
6,historical_data_2000.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2000Q3.zip,2000,Q3,historical_data_2000Q3.txt,historical_data_time_2000Q3.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
7,historical_data_2000.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2000Q4.zip,2000,Q4,historical_data_2000Q4.txt,historical_data_time_2000Q4.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
8,historical_data_2001.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2001Q1.zip,2001,Q1,historical_data_2001Q1.txt,historical_data_time_2001Q1.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...
9,historical_data_2001.zip,/content/drive/MyDrive/master_thesis/data/hist...,historical_data_2001Q2.zip,2001,Q2,historical_data_2001Q2.txt,historical_data_time_2001Q2.txt,/content/drive/MyDrive/master_thesis/data/parq...,/content/drive/MyDrive/master_thesis/data/parq...


Total quarter ZIPs  : 107
Year range          : 1999 – 2025

Quarters per year:
year
1999    4
2000    4
2001    4
2002    4
2003    4
2004    4
2005    4
2006    4
2007    4
2008    4
2009    4
2010    4
2011    4
2012    4
2013    4
2014    4
2015    4
2016    4
2017    4
2018    4
2019    4
2020    4
2021    4
2022    4
2023    4
2024    4
2025    3


---
## Section 10 · Ingestion Pipeline

The pipeline iterates year-by-year over the manifest. For each quarter:

1. Open the outer yearly ZIP.
2. Read the inner quarterly ZIP entirely into RAM.
3. Extract origination and performance TXTs to local NVMe (`TEMP_ROOT`).
4. Write typed Parquet files via DuckDB `COPY … TO` to Drive.
5. Delete the extracted TXTs immediately after writing.


In [ ]:
def run_ingestion(manifest: pd.DataFrame) -> pd.DataFrame:
    """
    Execute the full ingestion pipeline over all quarters in the manifest.
    Returns a DataFrame summarising the status of each quarter.
    """
    con     = connect_duckdb()
    results = []

    try:
        for year, year_df in tqdm(
            list(manifest.groupby("year", sort=True)),
            desc="Ingesting years", unit="year",
        ):
            year_zip_path = Path(year_df["source_year_zip_path"].iloc[0])

            with zipfile.ZipFile(year_zip_path, "r") as year_zip:
                for rec in tqdm(
                    year_df.to_dict(orient="records"),
                    desc=f"  {year}", unit="quarter", leave=False,
                ):
                    vy       = int(rec["year"])
                    vq       = rec["quarter"]
                    orig_out = Path(rec["orig_out"])
                    perf_out = Path(rec["perf_out"])

                    if orig_out.exists() and perf_out.exists() and not OVERWRITE_EXISTING:
                        results.append({"year": vy, "quarter": vq, "status": "skipped"})
                        continue

                    # Read inner quarter ZIP into RAM; extract TXTs to local NVMe
                    with year_zip.open(rec["source_quarter_zip"]) as inner_f:
                        inner_bytes = inner_f.read()

                    temp_q_dir = TEMP_ROOT / f"{vy}{vq}"
                    temp_q_dir.mkdir(parents=True, exist_ok=True)
                    orig_tmp = perf_tmp = None

                    try:
                        with zipfile.ZipFile(io.BytesIO(inner_bytes), "r") as qzip:
                            orig_tmp = extract_member_to_temp(qzip, rec["orig_txt"], temp_q_dir)
                            perf_tmp = extract_member_to_temp(qzip, rec["perf_txt"], temp_q_dir)
                        del inner_bytes

                        t0 = time.time()
                        ingest_txt_with_duckdb(
                            con, orig_tmp, orig_out, "origination",
                            vy, vq, rec["source_year_zip"],
                            rec["source_quarter_zip"], rec["orig_txt"],
                        )
                        ingest_txt_with_duckdb(
                            con, perf_tmp, perf_out, "performance",
                            vy, vq, rec["source_year_zip"],
                            rec["source_quarter_zip"], rec["perf_txt"],
                        )
                        results.append({
                            "year": vy, "quarter": vq, "status": "ok",
                            "elapsed_s": round(time.time() - t0, 1),
                        })

                    except Exception as exc:
                        results.append({
                            "year": vy, "quarter": vq, "status": "error", "error": str(exc)
                        })
                        print(f"\n  ✗ {vy}{vq}: {exc}")

                    finally:
                        for tmp in [orig_tmp, perf_tmp]:
                            if tmp is not None and tmp.exists():
                                tmp.unlink()
                        if temp_q_dir.exists() and not any(temp_q_dir.iterdir()):
                            temp_q_dir.rmdir()
    finally:
        con.close()

    return pd.DataFrame(results)

---
## Section 11 · Execute Ingestion

Run the full ingestion pipeline.


In [ ]:
ingestion_results = run_ingestion(manifest)

ok_n   = (ingestion_results["status"] == "ok").sum()
skip_n = (ingestion_results["status"] == "skipped").sum()
err_n  = (ingestion_results["status"] == "error").sum()

print(f"{'─'*50}")
print(f"  Processed  : {ok_n:>4}  quarters (newly written)")
print(f"  Skipped    : {skip_n:>4}  quarters (already on Drive)")
print(f"  Errors     : {err_n:>4}")

if err_n:
    print("\nFailed quarters:")
    print(ingestion_results[ingestion_results["status"] == "error"].to_string(index=False))

Ingesting years:   0%|          | 0/27 [00:00<?, ?year/s]

  1999:   0%|          | 0/4 [00:00<?, ?quarter/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2024:   0%|          | 0/4 [00:00<?, ?quarter/s]

  2025:   0%|          | 0/3 [00:00<?, ?quarter/s]

──────────────────────────────────────────────────
  Processed  :  107  quarters (newly written)
  Skipped    :    0  quarters (already on Drive)
  Errors     :    0


---
## Section 12 · Verify Output Files

Counts generated Parquet files and asserts that counts match the manifest.


In [ ]:
orig_files = sorted(ORIGINATION_OUT.rglob("*.parquet"))
perf_files = sorted(PERFORMANCE_OUT.rglob("*.parquet"))
n_quarters = len(manifest)

print(f"Origination Parquet files : {len(orig_files)}  (expected {n_quarters})")
print(f"Performance Parquet files : {len(perf_files)}  (expected {n_quarters})")

assert len(orig_files) == n_quarters, (
    f"Origination count mismatch: expected {n_quarters}, found {len(orig_files)}"
)
assert len(perf_files) == n_quarters, (
    f"Performance count mismatch: expected {n_quarters}, found {len(perf_files)}"
)
print(f"\n✓  File counts match manifest ({n_quarters} quarters).")

Origination Parquet files : 107  (expected 107)
Performance Parquet files : 107  (expected 107)

✓  File counts match manifest (107 quarters).


---
## Section 13 · Row Count Validation

Queries all Parquet files to confirm total ingested rows and compares them with
Freddie Mac's approximate Release 46 totals. The 5% threshold below is a coarse
gross-error diagnostic. Exact file-count
and repartition-parity checks are performed separately in §§12 and 16.


In [ ]:
con = connect_duckdb()

orig_glob = (ORIGINATION_OUT / "**" / "*.parquet").as_posix()
perf_glob = (PERFORMANCE_OUT  / "**" / "*.parquet").as_posix()

orig_count = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{orig_glob}', union_by_name=True)"
).fetchone()[0]
perf_count = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{perf_glob}', union_by_name=True)"
).fetchone()[0]

con.close()

ref_orig_m = SFLLD_RELEASE_METADATA["approx_origination_records_millions"]
ref_perf_b = SFLLD_RELEASE_METADATA["approx_performance_records_billions"]

print(f"Origination rows ingested : {orig_count:>15,}  ({orig_count/1e6:.2f} M)")
print(f"Performance rows ingested : {perf_count:>15,}  ({perf_count/1e9:.2f} B)")
print()
print(f"Release 46 reference      : ~{ref_orig_m:.2f} M origination,  ~{ref_perf_b:.2f} B performance")

orig_dev = abs(orig_count / 1e6 - ref_orig_m) / ref_orig_m * 100
perf_dev = abs(perf_count / 1e9 - ref_perf_b) / ref_perf_b * 100

# Coarse gross-error threshold against the approximate release totals;
# this is not a validated acceptance criterion.
ROWCOUNT_DEVIATION_TOL_PCT = 5.0

flag = lambda d: (
    f"✓  (within {ROWCOUNT_DEVIATION_TOL_PCT:.0f}% tolerance)"
    if d < ROWCOUNT_DEVIATION_TOL_PCT
    else f"⚠  DEVIATION ≥ {ROWCOUNT_DEVIATION_TOL_PCT:.0f}% - investigate"
)
print(f"\n  Origination deviation: {orig_dev:.1f}%  {flag(orig_dev)}")
print(f"  Performance deviation: {perf_dev:.1f}%  {flag(perf_dev)}")

Origination rows ingested :      48,597,637  (48.60 M)
Performance rows ingested :   2,800,558,227  (2.80 B)

Release 46 reference      : ~48.61 M origination,  ~2.81 B performance

  Origination deviation: 0.0%  ✓  (within 5% tolerance)
  Performance deviation: 0.3%  ✓  (within 5% tolerance)


---
## Section 14 · Schema Spot-Check

Confirms that the disclosure-sensitive fields documented in §1 were parsed with the intended types.


In [ ]:
con = connect_duckdb()

orig_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{orig_glob}', union_by_name=True) LIMIT 0"
).df().set_index("column_name")["column_type"]

perf_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{perf_glob}', union_by_name=True) LIMIT 0"
).df().set_index("column_name")["column_type"]

con.close()

checks = [
    ("postal_code",                     orig_schema, "VARCHAR"),
    ("estimated_loan_to_value",         perf_schema, "DOUBLE"),
    ("net_sale_proceeds",               perf_schema, "VARCHAR"),
    ("zero_balance_code",               perf_schema, "VARCHAR"),
    ("current_loan_delinquency_status", perf_schema, "VARCHAR"),
]

all_ok = True
for col, schema, expected_type in checks:
    actual = schema.get(col, "NOT FOUND")
    ok     = actual == expected_type
    all_ok = all_ok and ok
    print(f"  {'✓' if ok else '✗'} {col:<45} : {actual:<10} (expected {expected_type})")
    if not ok:
        print(f"    See §1 for the parsing-decision rationale for {col}.")

print()
print("✓  All schema checks passed." if all_ok else "⚠  Schema check failures - review SELECT builders.")

  ✓ postal_code                                   : VARCHAR    (expected VARCHAR)
  ✓ estimated_loan_to_value                       : DOUBLE     (expected DOUBLE)
  ✓ net_sale_proceeds                             : VARCHAR    (expected VARCHAR)
  ✓ zero_balance_code                             : VARCHAR    (expected VARCHAR)
  ✓ current_loan_delinquency_status               : VARCHAR    (expected VARCHAR)

✓  All schema checks passed.


---
## Section 15 · Repartition Performance Data by Reporting Period

The ingestion pipeline (§10) organises performance files by **origination vintage**
(`vintage_year / vintage_quarter`), matching the source layout. Downstream
raw-performance queries are mainly calendar-time queries on
`monthly_reporting_period`. In the vintage layout, reporting year is not a
directory partition key, so reporting-year filters cannot prune files by
partition, although Parquet filter pushdown may still skip row groups.

This section creates `parquet/performance_by_period/reporting_year=YYYY/`
with DuckDB `PARTITION_BY`. Reporting-year filters can then skip irrelevant
year files directly. The vintage-partitioned files are not modified and
remain the authoritative source.


In [ ]:
PERF_BY_PERIOD_LOCAL = LOCAL_ROOT / "parquet_by_period_local"   # NVMe staging
PERF_BY_PERIOD_DRIVE = PARQUET_ROOT / "performance_by_period"    # permanent Drive output

PERF_BY_PERIOD_LOCAL.mkdir(parents=True, exist_ok=True)
PERF_BY_PERIOD_DRIVE.mkdir(parents=True, exist_ok=True)

existing_drive_years = sorted({
    p.parent.name for p in PERF_BY_PERIOD_DRIVE.rglob("*.parquet")
})
print(f"Existing reporting-year partitions on Drive : {len(existing_drive_years)}")
if existing_drive_years:
    print(f"  e.g. {existing_drive_years[:4]} … {existing_drive_years[-2:]}")

Existing reporting-year partitions on Drive : 0


In [ ]:
# ── Repartition into staging ──────────────────────────────────────
drive_years = sorted({p.parent.name for p in PERF_BY_PERIOD_DRIVE.rglob("*.parquet")})
local_years = sorted({p.parent.name for p in PERF_BY_PERIOD_LOCAL.rglob("*.parquet")})

if drive_years:
    print(f"⏭  Repartition already present at {PERF_BY_PERIOD_DRIVE} "
          f"({len(drive_years)} year folders). Skipping Phase 1.")
    local_years = []
elif local_years:
    print(f"⏭  Staging repartition already complete ({len(local_years)} year folders). Skipping Phase 1.")
else:
    print("Phase 1 - full table scan of vintage performance parquets → staging repartition")
    print(f"  Source : {PERFORMANCE_OUT}")
    print(f"  Dest   : {PERF_BY_PERIOD_LOCAL} (staging)")

    con = connect_duckdb()
    t0  = time.time()

    con.execute(f"""
        COPY (
            SELECT
                *,
                YEAR(monthly_reporting_period) AS reporting_year
            FROM read_parquet('{perf_glob}', union_by_name=True)
        )
        TO '{PERF_BY_PERIOD_LOCAL.as_posix()}'
        (
            FORMAT              PARQUET,
            PARTITION_BY        (reporting_year),
            COMPRESSION         {PARQUET_COMPRESSION},
            ROW_GROUP_SIZE      {PARQUET_ROW_GROUP_SIZE},
            OVERWRITE_OR_IGNORE TRUE
        )
    """)
    con.close()

    elapsed   = time.time() - t0
    local_years = sorted({p.parent.name for p in PERF_BY_PERIOD_LOCAL.rglob("*.parquet")})
    local_files = list(PERF_BY_PERIOD_LOCAL.rglob("*.parquet"))
    local_gb    = sum(f.stat().st_size for f in local_files) / 1e9

    print(f"\n✓  Phase 1 complete in {elapsed/60:.1f} min")
    print(f"   Reporting-year partitions : {len(local_years)}")
    print(f"   Total Parquet files       : {len(local_files)}")
    print(f"   Total on-disk size        : {local_gb:.1f} GB")

Phase 1 - full table scan of vintage performance parquets → staging repartition
  Source : /content/drive/MyDrive/master_thesis/data/parquet/performance
  Dest   : /content/parquet_by_period_local (staging)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓  Phase 1 complete in 4.6 min
   Reporting-year partitions : 27
   Total Parquet files       : 27
   Total on-disk size        : 25.5 GB


In [ ]:
# ── Sync local NVMe → Drive, then clean up ─────────────────────────
# Determine which year folders still need to reach Drive
missing = [
    yr for yr in local_years
    if not list((PERF_BY_PERIOD_DRIVE / yr).glob("*.parquet"))
]

if missing:
    print(f"Phase 2 - syncing {len(missing)} year folder(s) to Drive …")
    t0 = time.time()
    for yr_folder in tqdm(missing, desc="Syncing to Drive", unit="year"):
        src = PERF_BY_PERIOD_LOCAL / yr_folder
        dst = PERF_BY_PERIOD_DRIVE / yr_folder
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.glob("*.parquet"):
            shutil.copy2(f, dst / f.name)

    # Clean up local staging area to reclaim NVMe space
    shutil.rmtree(PERF_BY_PERIOD_LOCAL, ignore_errors=True)
    print(f"\n✓  Phase 2 complete in {(time.time()-t0)/60:.1f} min - local staging cleaned.")
else:
    print("✓  All year folders already on Drive.")
    shutil.rmtree(PERF_BY_PERIOD_LOCAL, ignore_errors=True)

# ── Verification ─────────────────────────────────────────────────────────────
period_files = sorted(PERF_BY_PERIOD_DRIVE.rglob("*.parquet"))
period_years = sorted({p.parent.name for p in period_files})
perf_by_period_glob = (PERF_BY_PERIOD_DRIVE / "**" / "*.parquet").as_posix()

print(f"Reporting-year partitions on Drive : {len(period_years)}")
print(f"Total Parquet files                : {len(period_files)}")

✓  All year folders already on Drive.
Reporting-year partitions on Drive : 27
Total Parquet files                : 27


---
## Section 16 · Final Row Count Validation & Ingestion Manifest

Confirms that the repartitioned row count matches the vintage-partitioned
row count exactly.

Then writes `ingestion_manifest.json`, which NB02 and NB03 read to report
the upstream ingestion snapshot.


In [ ]:
con = connect_duckdb()

period_count  = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{perf_by_period_glob}', union_by_name=True)"
).fetchone()[0]
vintage_count = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{perf_glob}', union_by_name=True)"
).fetchone()[0]

con.close()

assert period_count == vintage_count, (
    f"Row count mismatch after repartition: "
    f"vintage={vintage_count:,}, by-period={period_count:,}"
)
print(f"✓  Performance row counts match: {period_count:,} rows")
print(f"   Vintage-partitioned  : {vintage_count:,}")
print(f"   Period-partitioned   : {period_count:,}")

# ── Write ingestion manifest ─────────────────────────────────────────────────
ingestion_manifest = {
    "release_number"          : SFLLD_RELEASE_METADATA["release_number"],
    "release_date"            : SFLLD_RELEASE_METADATA["release_date"],
    "origination_cutoff_date" : SFLLD_RELEASE_METADATA["origination_cutoff_date"],
    "performance_cutoff_date" : SFLLD_RELEASE_METADATA["performance_cutoff_date"],
    "dataset"                 : SFLLD_RELEASE_METADATA["dataset"],
    "total_origination_rows"  : int(orig_count),
    "total_performance_rows"  : int(perf_count),
}

manifest_path = MANIFEST_DIR / "ingestion_manifest.json"
with open(manifest_path, "w") as fh:
    json.dump(ingestion_manifest, fh, indent=2)

print(f"\n✓  Ingestion manifest written: {manifest_path}")
print(f"   Release : {ingestion_manifest['release_number']} ({ingestion_manifest['release_date']})")
print(f"   Orig    : {orig_count:,} rows  ({orig_count/1e6:.2f} M)")
print(f"   Perf    : {perf_count:,} rows  ({perf_count/1e9:.2f} B)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  Performance row counts match: 2,800,558,227 rows
   Vintage-partitioned  : 2,800,558,227
   Period-partitioned   : 2,800,558,227

✓  Ingestion manifest written: /content/drive/MyDrive/master_thesis/manifests/ingestion_manifest.json
   Release : 46 (2026-01-28)
   Orig    : 48,597,637 rows  (48.60 M)
   Perf    : 2,800,558,227 rows  (2.80 B)


---

## Appendix · References

- Freddie Mac. (2026a, January). Single-family loan-level dataset frequently asked questions (FAQ) (Frequently Asked Questions). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

- Freddie Mac. (2026b, January). Single-family loan-level dataset general user guide (User Guide). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

- Freddie Mac. (2026c, January). Single-family loan-level dataset release notes (Release Notes). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset